# 📡 Investment Data Health Monitor
### Python · Pandas · Plotly · scikit-learn
**Author:** Anushree Revankar | B.Tech Computer Engineering (Hons. Data Science), DJSCE Mumbai

---
This notebook demonstrates a production-style **Data Health Monitoring pipeline** for critical investment data domains:
- 🏦 Securities Reference Data
- 📊 Positions
- 🔄 Trades
- 📈 Share Prices
- 👤 Client Data

**Covers:** Synthetic data generation · KPI validation · RAG thresholds · IQR anomaly detection · Exception logging · Data dictionary


## ⚙️ Step 0 — Install & Import Libraries

In [ ]:
!pip install plotly -q

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.ensemble import IsolationForest
from datetime import datetime, timedelta
import random
import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries loaded successfully")
print(f"📅 Run timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


## 🧪 Step 1 — Synthetic Investment Data Generation
Simulates realistic investment data across 5 domains using `numpy` and `pandas`.  
In production, this would be replaced by queries to Snowflake / Databricks / Bloomberg API.


In [ ]:
np.random.seed(42)
random.seed(42)
N_SECURITIES = 200
N_POSITIONS  = 500
N_TRADES     = 300
N_PRICES     = 150
N_CLIENTS    = 100

ASSET_CLASSES  = ['Equity', 'Fixed Income', 'Derivative', 'FX', 'Commodity']
EXCHANGES      = ['NSE', 'BSE', 'NYSE', 'LSE', 'HKEX']
CURRENCIES     = ['USD', 'INR', 'GBP', 'EUR', 'JPY']
PRICE_SOURCES  = ['Bloomberg', 'Refinitiv', 'ICE', 'SIX']
KYC_STATUSES   = ['Verified', 'Pending', 'Expired']
TRADE_TYPES    = ['Buy', 'Sell', 'Short', 'Cover']
TRADE_STATUSES = ['Settled', 'Pending', 'Failed', 'Cancelled']

def random_isin():
    return 'IN' + ''.join([str(random.randint(0,9)) for _ in range(10)])

def inject_nulls(series, pct=0.06):
    mask = np.random.random(len(series)) < pct
    s = series.astype(object)
    s[mask] = np.nan
    return s

today = datetime.today()

# Securities
securities = pd.DataFrame({
    'isin':          [random_isin() for _ in range(N_SECURITIES)],
    'security_name': inject_nulls(pd.array([f'Security {i:04d} Ltd' for i in range(N_SECURITIES)])),
    'asset_class':   np.random.choice(ASSET_CLASSES, N_SECURITIES),
    'exchange':      inject_nulls(pd.array(np.random.choice(EXCHANGES, N_SECURITIES))),
    'currency':      np.random.choice(CURRENCIES, N_SECURITIES),
    'issuer':        inject_nulls(pd.array([f'Issuer {i}' for i in range(N_SECURITIES)])),
    'last_updated':  [today - timedelta(days=random.randint(0,5)) for _ in range(N_SECURITIES)],
    'is_active':     np.random.choice([True, False], N_SECURITIES, p=[0.92, 0.08]),
})

# Positions
portfolio_ids = [f'PORT{i:04d}' for i in range(50)]
positions = pd.DataFrame({
    'portfolio_id':  np.random.choice(portfolio_ids, N_POSITIONS),
    'isin':          np.random.choice(securities['isin'].tolist(), N_POSITIONS),
    'quantity':      inject_nulls(pd.array(np.round(np.random.uniform(100, 100000, N_POSITIONS), 2))),
    'market_value':  inject_nulls(pd.array(np.round(np.random.uniform(10000, 5000000, N_POSITIONS), 2))),
    'currency':      np.random.choice(CURRENCIES, N_POSITIONS),
    'as_of_date':    [today - timedelta(days=random.randint(0,3)) for _ in range(N_POSITIONS)],
    'cost_basis':    inject_nulls(pd.array(np.round(np.random.uniform(5000, 4500000, N_POSITIONS), 2))),
})

# Trades
trades = pd.DataFrame({
    'trade_id':        [f'TRD{i:06d}' for i in range(N_TRADES)],
    'isin':            np.random.choice(securities['isin'].tolist(), N_TRADES),
    'portfolio_id':    np.random.choice(portfolio_ids, N_TRADES),
    'trade_type':      np.random.choice(TRADE_TYPES, N_TRADES),
    'quantity':        inject_nulls(pd.array(np.round(np.random.uniform(10, 50000, N_TRADES), 0))),
    'price':           inject_nulls(pd.array(np.round(np.random.uniform(10, 5000, N_TRADES), 4))),
    'trade_date':      [today - timedelta(days=random.randint(0,10)) for _ in range(N_TRADES)],
    'settlement_date': inject_nulls(pd.array([today + timedelta(days=random.choice([-1,0,1,2,3])) for _ in range(N_TRADES)])),
    'status':          np.random.choice(TRADE_STATUSES, N_TRADES, p=[0.7,0.15,0.1,0.05]),
    'broker':          inject_nulls(pd.array([f'Broker {random.randint(1,10)}' for _ in range(N_TRADES)])),
})

# Share Prices — with deliberate outliers for anomaly detection
base_prices = np.random.uniform(50, 2000, N_PRICES)
price_noise = np.random.normal(0, base_prices * 0.02, N_PRICES)
prices_raw  = base_prices + price_noise
outlier_idx = np.random.choice(N_PRICES, size=8, replace=False)
prices_raw[outlier_idx] *= np.random.choice([1.25, 0.75], size=8)

share_prices = pd.DataFrame({
    'isin':         np.random.choice(securities['isin'].tolist(), N_PRICES),
    'price_date':   [today - timedelta(days=random.randint(0,2)) for _ in range(N_PRICES)],
    'close_price':  inject_nulls(pd.array(np.round(prices_raw, 4))),
    'open_price':   inject_nulls(pd.array(np.round(base_prices + np.random.normal(0, base_prices*0.01, N_PRICES), 4))),
    'volume':       inject_nulls(pd.array(np.random.randint(1000, 10000000, N_PRICES))),
    'price_source': inject_nulls(pd.array(np.random.choice(PRICE_SOURCES, N_PRICES))),
    'currency':     np.random.choice(CURRENCIES, N_PRICES),
    'is_stale':     np.random.choice([False, True], N_PRICES, p=[0.93, 0.07]),
    '_is_outlier':  [i in outlier_idx for i in range(N_PRICES)],  # ground truth label
})

# Client Data
client_data = pd.DataFrame({
    'client_id':      [f'CLI{i:05d}' for i in range(N_CLIENTS)],
    'client_name':    inject_nulls(pd.array([f'Client {i}' for i in range(N_CLIENTS)])),
    'kyc_status':     np.random.choice(KYC_STATUSES, N_CLIENTS, p=[0.75, 0.15, 0.10]),
    'onboard_date':   [today - timedelta(days=random.randint(30, 2000)) for _ in range(N_CLIENTS)],
    'relationship_mgr': inject_nulls(pd.array([f'RM {random.randint(1,15)}' for _ in range(N_CLIENTS)])),
    'aum_usd':        inject_nulls(pd.array(np.round(np.random.uniform(100000, 50000000, N_CLIENTS), 2))),
    'email':          inject_nulls(pd.array([f'client{i}@example.com' for i in range(N_CLIENTS)])),
    'risk_profile':   np.random.choice(['Conservative','Moderate','Aggressive'], N_CLIENTS),
})

print("✅ Synthetic data generated")
print(f"   Securities:   {len(securities):>5,} rows")
print(f"   Positions:    {len(positions):>5,} rows")
print(f"   Trades:       {len(trades):>5,} rows")
print(f"   Share Prices: {len(share_prices):>5,} rows")
print(f"   Client Data:  {len(client_data):>5,} rows")


## ✅ Step 2 — KPI Validation Engine
Computes **Completeness, Timeliness, Validity, Recon Breaks** for each domain.  
RAG thresholds: 🟢 ≥95% · 🟡 80–95% · 🔴 <80%


In [ ]:
SLA_DAYS = 2  # max acceptable staleness in days

def completeness(df, exclude_cols=None):
    cols = [c for c in df.columns if c not in (exclude_cols or ['_is_outlier'])]
    return round(df[cols].notna().mean().mean() * 100, 2)

def timeliness(df, date_col, sla_days=SLA_DAYS):
    if date_col not in df.columns: return 100.0
    stale = (today - pd.to_datetime(df[date_col])).dt.days > sla_days
    return round((~stale).mean() * 100, 2)

def validity_securities(df):
    valid_currency  = df['currency'].isin(CURRENCIES)
    valid_active    = df['is_active'].isin([True, False])
    return round(pd.concat([valid_currency, valid_active], axis=1).all(axis=1).mean() * 100, 2)

def validity_trades(df):
    valid_type   = df['trade_type'].isin(TRADE_TYPES)
    valid_status = df['status'].isin(TRADE_STATUSES)
    valid_qty    = df['quantity'].fillna(0) > 0
    return round(pd.concat([valid_type, valid_status, valid_qty], axis=1).all(axis=1).mean() * 100, 2)

def validity_prices(df):
    valid_price  = df['close_price'].fillna(-1) > 0
    valid_source = df['price_source'].isin(PRICE_SOURCES)
    return round(pd.concat([valid_price, valid_source], axis=1).all(axis=1).mean() * 100, 2)

def validity_clients(df):
    valid_kyc   = df['kyc_status'].isin(KYC_STATUSES)
    valid_email = df['email'].str.contains('@', na=False)
    return round(pd.concat([valid_kyc, valid_email], axis=1).all(axis=1).mean() * 100, 2)

def recon_breaks_positions(pos, sec):
    merged = pos.merge(sec[['isin']], on='isin', how='left', indicator=True)
    breaks = (merged['_merge'] == 'left_only').sum()
    return int(breaks)

def rag(value, green=95, amber=80):
    if value >= green: return '🟢 Healthy'
    elif value >= amber: return '🟡 At Risk'
    return '🔴 Critical'

kpis = {
    'Securities': {
        'Completeness (%)': completeness(securities),
        'Timeliness (%)':   timeliness(securities, 'last_updated'),
        'Validity (%)':     validity_securities(securities),
        'Recon Breaks':     0,
    },
    'Positions': {
        'Completeness (%)': completeness(positions),
        'Timeliness (%)':   timeliness(positions, 'as_of_date'),
        'Validity (%)':     round(positions['quantity'].fillna(0).gt(0).mean()*100, 2),
        'Recon Breaks':     recon_breaks_positions(positions, securities),
    },
    'Trades': {
        'Completeness (%)': completeness(trades),
        'Timeliness (%)':   timeliness(trades, 'trade_date'),
        'Validity (%)':     validity_trades(trades),
        'Recon Breaks':     int((trades['status'] == 'Failed').sum()),
    },
    'Share Prices': {
        'Completeness (%)': completeness(share_prices, exclude_cols=['_is_outlier']),
        'Timeliness (%)':   round((~share_prices['is_stale']).mean()*100, 2),
        'Validity (%)':     validity_prices(share_prices),
        'Recon Breaks':     int(share_prices['is_stale'].sum()),
    },
    'Client Data': {
        'Completeness (%)': completeness(client_data),
        'Timeliness (%)':   100.0,
        'Validity (%)':     validity_clients(client_data),
        'Recon Breaks':     int((client_data['kyc_status'] == 'Expired').sum()),
    },
}

kpi_df = pd.DataFrame(kpis).T.reset_index().rename(columns={'index': 'Domain'})
kpi_df['Overall RAG'] = kpi_df.apply(
    lambda r: '🔴 Critical' if any([r['Completeness (%)'] < 80, r['Timeliness (%)'] < 80, r['Validity (%)'] < 80])
              else ('🟡 At Risk' if any([r['Completeness (%)'] < 95, r['Timeliness (%)'] < 95, r['Validity (%)'] < 95])
              else '🟢 Healthy'), axis=1)

print("=" * 65)
print("INVESTMENT DATA HEALTH — KPI SUMMARY")
print("=" * 65)
print(kpi_df.to_string(index=False))
print("=" * 65)
print(f"Run at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


## 📊 Step 3 — RAG Status Dashboard (Interactive)

In [ ]:
COLORS = {'🟢 Healthy': '#16a34a', '🟡 At Risk': '#d97706', '🔴 Critical': '#dc2626'}
LIGHT  = {'🟢 Healthy': '#dcfce7', '🟡 At Risk': '#fef3c7', '🔴 Critical': '#fee2e2'}
DOMAINS_LIST = kpi_df['Domain'].tolist()

fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=('Completeness by Domain (%)', 'Timeliness by Domain (%)'),
    vertical_spacing=0.18
)

bar_colors_c = [COLORS.get(kpi_df.loc[kpi_df['Domain']==d, 'Overall RAG'].values[0], '#3b82f6') for d in DOMAINS_LIST]

fig.add_trace(go.Bar(
    x=DOMAINS_LIST, y=kpi_df['Completeness (%)'],
    marker_color=bar_colors_c, name='Completeness',
    text=[f"{v}%" for v in kpi_df['Completeness (%)']],
    textposition='outside', showlegend=False
), row=1, col=1)

fig.add_trace(go.Bar(
    x=DOMAINS_LIST, y=kpi_df['Timeliness (%)'],
    marker_color=bar_colors_c, name='Timeliness',
    text=[f"{v}%" for v in kpi_df['Timeliness (%)']],
    textposition='outside', showlegend=False
), row=2, col=1)

for row in [1, 2]:
    fig.add_hline(y=95, line_dash='dash', line_color='#16a34a', annotation_text='Green threshold (95%)', row=row, col=1)
    fig.add_hline(y=80, line_dash='dash', line_color='#d97706', annotation_text='Amber threshold (80%)', row=row, col=1)

fig.update_yaxes(range=[60, 105])
fig.update_layout(
    title='📡 Investment Data Health Dashboard',
    height=550, template='plotly_white',
    font=dict(family='DM Sans, sans-serif'),
    margin=dict(t=80, b=40)
)
fig.show()

# RAG Summary Table
summary = kpi_df[['Domain', 'Completeness (%)', 'Timeliness (%)', 'Validity (%)', 'Recon Breaks', 'Overall RAG']].copy()
print("\n📋 DOMAIN RAG SUMMARY")
print(summary.to_string(index=False))


## 🚨 Step 4 — Anomaly Detection on Share Prices
Two methods compared:
- **IQR Method** — simple, interpretable, no training needed
- **Isolation Forest** (scikit-learn) — ML-based, better for multivariate data


In [ ]:
prices_clean = share_prices.dropna(subset=['close_price']).copy().reset_index(drop=True)

# ── Method 1: IQR ──────────────────────────────────────────────────────────────
Q1, Q3 = prices_clean['close_price'].quantile(0.25), prices_clean['close_price'].quantile(0.75)
IQR = Q3 - Q1
lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR
prices_clean['iqr_anomaly'] = (prices_clean['close_price'] < lower_fence) | (prices_clean['close_price'] > upper_fence)

# ── Method 2: Isolation Forest ────────────────────────────────────────────────
features = prices_clean[['close_price']].fillna(prices_clean['close_price'].median())
iso = IsolationForest(contamination=0.07, random_state=42)
prices_clean['iso_anomaly'] = iso.fit_predict(features) == -1

# ── Agreement ─────────────────────────────────────────────────────────────────
prices_clean['any_anomaly'] = prices_clean['iqr_anomaly'] | prices_clean['iso_anomaly']
prices_clean['both_anomaly'] = prices_clean['iqr_anomaly'] & prices_clean['iso_anomaly']

print(f"IQR bounds:         [${lower_fence:.2f}  –  ${upper_fence:.2f}]")
print(f"IQR anomalies:       {prices_clean['iqr_anomaly'].sum()}")
print(f"Isolation Forest:    {prices_clean['iso_anomaly'].sum()}")
print(f"Flagged by both:     {prices_clean['both_anomaly'].sum()}")
print(f"Ground truth:        {prices_clean['_is_outlier'].sum()} injected outliers")

# ── Plot ───────────────────────────────────────────────────────────────────────
fig2 = go.Figure()
normal = prices_clean[~prices_clean['any_anomaly']]
anomaly = prices_clean[prices_clean['any_anomaly']]

fig2.add_trace(go.Scatter(
    x=normal.index, y=normal['close_price'],
    mode='markers', name='Normal', marker=dict(color='#3b82f6', size=6, opacity=0.6)
))
fig2.add_trace(go.Scatter(
    x=anomaly.index, y=anomaly['close_price'],
    mode='markers', name='Anomaly', marker=dict(color='#dc2626', size=10, symbol='x', line=dict(width=2))
))
fig2.add_hline(y=upper_fence, line_dash='dot', line_color='#d97706', annotation_text=f'Upper fence (IQR): ${upper_fence:.2f}')
fig2.add_hline(y=lower_fence, line_dash='dot', line_color='#d97706', annotation_text=f'Lower fence (IQR): ${lower_fence:.2f}')
fig2.add_hline(y=Q1, line_dash='dash', line_color='#94a3b8', annotation_text='Q1')
fig2.add_hline(y=Q3, line_dash='dash', line_color='#94a3b8', annotation_text='Q3')

fig2.update_layout(
    title='🔍 Share Price Anomaly Detection — IQR + Isolation Forest',
    xaxis_title='Security Index', yaxis_title='Close Price (USD)',
    height=450, template='plotly_white', font=dict(family='DM Sans, sans-serif')
)
fig2.show()

print("\n🚨 FLAGGED ANOMALIES")
flagged = prices_clean[prices_clean['any_anomaly']][['isin','close_price','price_source','iqr_anomaly','iso_anomaly']].head(10)
print(flagged.to_string(index=False))


## 📋 Step 5 — Exception Log & AI-Style Summary

In [ ]:
exceptions = []

for _, row in kpi_df.iterrows():
    d = row['Domain']
    if row['Completeness (%)'] < 95:
        sev = 'CRITICAL' if row['Completeness (%)'] < 80 else 'WARNING'
        exceptions.append({'Domain':d, 'Type':'Completeness', 'Severity':sev,
            'Value':f"{row['Completeness (%)']}%",
            'Message':f"{d} completeness at {row['Completeness (%)']}% — below {'critical (80%)' if sev=='CRITICAL' else 'warning (95%)'} threshold."})
    if row['Timeliness (%)'] < 90:
        exceptions.append({'Domain':d, 'Type':'SLA Breach', 'Severity':'CRITICAL',
            'Value':f"{row['Timeliness (%)']}%",
            'Message':f"{d} timeliness SLA breached. Current: {row['Timeliness (%)']}%, required ≥90%."})
    if row['Recon Breaks'] > 5:
        sev = 'CRITICAL' if row['Recon Breaks'] > 15 else 'WARNING'
        exceptions.append({'Domain':d, 'Type':'Recon Breaks', 'Severity':sev,
            'Value':str(int(row['Recon Breaks'])),
            'Message':f"{int(row['Recon Breaks'])} unresolved breaks in {d}. Investigate source alignment."})

n_price_anomalies = int(prices_clean['any_anomaly'].sum())
if n_price_anomalies > 0:
    exceptions.append({'Domain':'Share Prices','Type':'Anomaly','Severity':'WARNING',
        'Value':f"{n_price_anomalies} prices",
        'Message':f"{n_price_anomalies} price anomalies detected via IQR + Isolation Forest. Validate vendor feed."})

exc_df = pd.DataFrame(exceptions)
critical = exc_df[exc_df['Severity']=='CRITICAL']
warnings_ = exc_df[exc_df['Severity']=='WARNING']

print("=" * 65)
print("EXCEPTION LOG")
print("=" * 65)
print(f"{'CRITICAL':>10}: {len(critical)}")
print(f"{'WARNING':>10}: {len(warnings_)}")
print("=" * 65)
for _, r in exc_df.iterrows():
    icon = "🔴" if r['Severity']=='CRITICAL' else "🟡"
    print(f"\n{icon} [{r['Severity']}] {r['Domain']} — {r['Type']} ({r['Value']})")
    print(f"   {r['Message']}")

# AI-style summary
n_crit = len(critical)
print("\n" + "=" * 65)
print("🤖 AUTO-GENERATED EXCEPTION SUMMARY")
print("=" * 65)
if n_crit > 0:
    crit_domains = critical['Domain'].unique().tolist()
    print(f"{n_crit} critical issue(s) require immediate attention in: {', '.join(crit_domains)}.")
    print(f"Prioritise SLA breach resolution and reconciliation breaks.")
    print(f"{n_price_anomalies} price anomalies flagged for vendor data validation.")
else:
    print("No critical issues. Monitor amber-status domains.")
    print(f"{len(warnings_)} warnings outstanding — review before next SLA checkpoint.")


---
## 🤖 AI Assistance & Validation Approach

This project used AI assistants (Claude, ChatGPT) at several stages. Below is a transparent account of **what AI generated, what was validated, and how**.

| Stage | AI Contribution | Validation Method |
|-------|----------------|------------------|
| Exception summary text | AI generated natural-language summaries from structured KPI values | Cross-checked every summary against the raw `kpi_df` DataFrame — e.g. if summary said 'SLA breached in Trades', confirmed `timeliness < 90` in the table |
| Anomaly description | AI drafted the per-outlier explanation ('price above/below expected range') | Verified direction (above vs below) against computed `upper_fence` and `lower_fence` values programmatically |
| Data dictionary definitions | AI suggested field descriptions | Reviewed each definition against standard investment data conventions (Bloomberg field naming, ISDA terminology) |
| SOP checklist structure | AI proposed the RACI + threshold columns | Validated thresholds (95%/80%) against industry data quality benchmarks (DQMM, DCAM frameworks) |
| Code scaffolding | AI suggested pandas/sklearn patterns | Every function was unit-tested against expected outputs before inclusion |

### Validation principles applied
1. **No AI output was used verbatim without verification** — all summaries were spot-checked against the underlying data
2. **Quantitative claims were always re-derived** — if AI said '3 critical exceptions', the count was confirmed by `len(critical)` in code
3. **Domain knowledge check** — investment data concepts (T+2 settlement, ISIN format, KYC statuses) were verified against external references, not taken from AI alone
4. **Explainability preserved** — chosen models (IQR, Isolation Forest) were selected because their outputs can be explained to non-technical stakeholders, not just for accuracy

> *This approach mirrors production data management practice: AI accelerates drafting and pattern recognition, but a human-in-the-loop validates every output before it reaches stakeholders or drives decisions.*


## 📚 Step 6 — Data Dictionary

In [ ]:
data_dict = pd.DataFrame([
    ('Securities','isin','VARCHAR(12)','Ref Data Team','Daily 7AM','International Securities Identification Number'),
    ('Securities','security_name','VARCHAR(255)','Ref Data Team','Daily 7AM','Full legal name of the security'),
    ('Securities','asset_class','ENUM','Ref Data Team','Daily 7AM','Equity / Fixed Income / Derivative / FX'),
    ('Securities','exchange','VARCHAR(10)','Ref Data Team','Daily 7AM','Primary listing exchange code'),
    ('Securities','currency','CHAR(3)','Ref Data Team','Daily 7AM','ISO 4217 base currency of the security'),
    ('Positions','portfolio_id','VARCHAR(20)','Portfolio Ops','T+1 EOD','Unique identifier for client portfolio'),
    ('Positions','market_value','DECIMAL(18,4)','Portfolio Ops','T+1 EOD','MTM value in base currency (USD)'),
    ('Positions','quantity','DECIMAL(18,6)','Portfolio Ops','T+1 EOD','Number of units held'),
    ('Positions','as_of_date','DATE','Portfolio Ops','T+1 EOD','Valuation date for position snapshot'),
    ('Trades','trade_id','VARCHAR(36)','Trade Ops','Real-time','UUID for each trade execution'),
    ('Trades','trade_type','ENUM','Trade Ops','Real-time','Buy / Sell / Short / Cover'),
    ('Trades','settlement_date','DATE','Trade Ops','Real-time','Expected settlement date (T+2 equities)'),
    ('Trades','status','ENUM','Trade Ops','Real-time','Settled / Pending / Failed / Cancelled'),
    ('Share Prices','close_price','DECIMAL(12,4)','Market Data','EOD 6PM','Official closing price from primary exchange'),
    ('Share Prices','price_source','VARCHAR(50)','Market Data','EOD 6PM','Data vendor: Bloomberg, Refinitiv, ICE, SIX'),
    ('Share Prices','is_stale','BOOLEAN','Market Data','EOD 6PM','True if price not updated within SLA window'),
    ('Client Data','client_id','VARCHAR(20)','Client Services','On change','Unique client identifier in CRM'),
    ('Client Data','kyc_status','ENUM','Compliance','On change','Verified / Pending / Expired'),
    ('Client Data','aum_usd','DECIMAL(18,2)','Portfolio Ops','Monthly','Assets under management in USD'),
    ('Client Data','risk_profile','ENUM','Client Services','On change','Conservative / Moderate / Aggressive'),
], columns=['Domain','Field','Type','Owner','Refresh Cadence','Description'])

print("DATA DICTIONARY")
print("=" * 100)
for domain in data_dict['Domain'].unique():
    print(f"\n🏷️  {domain}")
    print("-" * 100)
    print(data_dict[data_dict['Domain']==domain][['Field','Type','Owner','Refresh Cadence','Description']].to_string(index=False))


## 📄 Step 7 — SOP Validation Checklist

In [ ]:
sop_checks = pd.DataFrame([
    ('Securities','Completeness','% non-null fields across all columns','≥95%','Ref Data Team','Daily 8AM','Ref Data Ops Lead'),
    ('Securities','Timeliness','Days since last_updated > SLA_DAYS','≤2 days','Ref Data Team','Daily 8AM','Ref Data Ops Lead'),
    ('Positions','Completeness','% non-null in quantity, market_value','≥95%','Portfolio Ops','T+1 EOD','Portfolio Ops Lead'),
    ('Positions','Recon Breaks','ISINs in Positions not in Securities master','0','Portfolio Ops','Daily 9AM','Reconciliation Team'),
    ('Trades','Validity','Trade type and status in approved enum sets','100%','Trade Ops','Real-time','Trade Ops Lead'),
    ('Trades','Failed Trades','Count of status=Failed','0','Trade Ops','Real-time','Trade Ops Lead'),
    ('Share Prices','Validity','close_price > 0 and source in approved vendors','≥95%','Market Data','EOD 6PM','Market Data Lead'),
    ('Share Prices','Anomaly','IQR or Isolation Forest outlier count','0','Market Data','EOD 6PM','Data Quality Team'),
    ('Client Data','KYC Expired','Count of kyc_status=Expired','0','Compliance','Weekly','Compliance Officer'),
], columns=['Domain','Check','Acceptance Criteria','Threshold','Owner','SLA','Escalation To'])

print("SOP VALIDATION CHECKLIST")
print("=" * 110)
print(sop_checks.to_string(index=False))
print("\n✅ All checks documented with business outcomes, acceptance criteria, RACI owners, and SLAs.")


## 🏁 Summary
This notebook demonstrated a complete **Investment Data Health Monitoring pipeline**:

| Step | Deliverable | JD Pillar |
|------|-------------|-----------|
| 1 | Synthetic data (Securities, Positions, Trades, Prices, Clients) | Data flow & lineage |
| 2 | KPI validation (Completeness, Timeliness, Validity, Recon Breaks) | Data health dashboard |
| 3 | Interactive RAG dashboard with Plotly | Data health dashboard |
| 4 | IQR + Isolation Forest anomaly detection on share prices | Augmented data management |
| 5 | Prioritised exception log with auto-generated summary | Automation |
| 6 | Data dictionary with ownership, refresh cadence, definitions | Data dictionary |
| 7 | SOP checklist with RACI, thresholds, SLAs | SOP updates |

**Production extensions:**
- Replace synthetic data with Snowflake / Databricks queries
- Schedule via Airflow or Azure Data Factory
- Push exception summaries via email/Teams webhook
- Add Gemini API for richer AI exception narration
